In [ ]:
from tensorflow.python.keras.utils import to_categorical
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D
import os
from keras_preprocessing.image import load_img
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import PIL
import cv2
from sklearn.preprocessing import LabelEncoder

In [114]:
train_dir='train'
test_dir='test'

In [115]:
def creatdataframe(dir):
    image_path = []
    labels = []  # Changed from 'label' to 'labels' to avoid confusion
    for label in os.listdir(dir):  # Assuming 'dir' contains subdirectories as class labels
        if os.path.isdir(os.path.join(dir, label)):  # Ensures only directories are processed
            for imagename in os.listdir(os.path.join(dir, label)):
                image_path.append(os.path.join(dir, label, imagename))
                labels.append(label)  # Fixed to use the list 'labels'
            print(f"{label} completed")
    return image_path, labels

In [116]:
train=pd.DataFrame()
train['image'], train['label'] = creatdataframe(train_dir)

angry completed
disgusted completed
fearful completed
happy completed
neutral completed
sad completed
surprised completed


In [117]:
test=pd.DataFrame()
test['image'], test['label'] = creatdataframe(test_dir)

angry completed
disgusted completed
fearful completed
happy completed
neutral completed
sad completed
surprised completed


In [118]:
#def extract_features(images):
 #   features = []
  #  for image in tqdm(images):
        # Use 'color_mode="grayscale"' instead of 'grayscale=True'
   #     img = load_img(image, color_mode='grayscale')
    #    img = np.array(img)
     #   features.append(img)
    #features = np.array(features)
    #features = features.reshape(len(features), 48, 48, 1)
    #return features

In [ ]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        
        img = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Unable to load image {image}. Skipping.")
            continue

        img = cv2.resize(img, (48, 48))
        img = np.expand_dims(img, axis=-1)
        features.append(img)

    features = np.array(features)
    print(f"Extracted features shape: {features.shape}")
    return features

In [100]:
train_features= extract_features(train['image'])

  0%|          | 0/28709 [00:00<?, ?it/s]

Extracted features shape: (28709, 48, 48, 1)


In [101]:
test_features= extract_features(test['image'])

  0%|          | 0/7178 [00:00<?, ?it/s]

Extracted features shape: (7178, 48, 48, 1)


In [120]:
x_train = train_features/255.0
x_test = test_features/255.0

In [121]:
le = LabelEncoder()
le.fit(train['label'])

LabelEncoder()

In [122]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

In [123]:
y_train = to_categorical(y_train, num_classes = 7)
y_test = to_categorical(y_test, num_classes = 7)

In [126]:
model = Sequential()
#layers
model.add(Conv2D(128, kernel_size = (3, 3), activation='relu', input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size = (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size = (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size = (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())
#fully connected layer
model.add(Dense(512, activation = 'relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
#output layer
model.add(Dense(7, activation='softmax'))

In [128]:
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

In [130]:
model.fit(x=x_train, y=y_train, batch_size=128, epochs=100, validation_data=(x_test,y_test))

Epoch 1/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1670s 7s/step - accuracy: 0.2433 - loss: 1.8362 - val_accuracy: 0.2471 - val_loss: 1.8187
Epoch 2/100
  7/225 ━━━━━━━━━━━━━━━━━━━━ 25:34 7s/step - accuracy: 0.2284 - loss: 1.8298


KeyboardInterrupt



In [ ]:
model_json = model.to_json()
with open("facialemotionmodel.json",'w') as json_file:
    json_file.write(model_json)
model.save("facialemotionmodel.h5")

In [ ]:
from tensorflow.python.keras.models import model_from_json

In [ ]:
json_file = open("facialemotionmodel.json",'r')
model_json = json_file.read()
json_file.close()
model = model_from_json(model_json)
model.load_weight("facialemotionmodel.h5")


In [ ]:
label = ['angry','disgust','fearful','happy','neutral','sad','surprised']


In [ ]:
def ef(image):
    img = load_img(image,greyscale = True)
    feature = np.array(img)
    feature = feature.reshape(1,48,48,1)
    return feature/255.0

